# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [2]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Eliminamos columnas identificativas o con texto poco útil
columnas_eliminar = ["PassengerId", "Name", "Cabin"]

# Separamos las variables predictoras y la variable objetivo
X = spaceship.drop(columns=columnas_eliminar + ["Transported"])
y = spaceship["Transported"]

# Identificamos variables numéricas y categóricas
columnas_numericas = X.select_dtypes(include=["number"]).columns.tolist()
columnas_categoricas = X.select_dtypes(
    include=["object", "bool"]
).columns.tolist()

# Tratamiento de variables numéricas
preprocesamiento_numerico = Pipeline([
    ("imputar", SimpleImputer(strategy="median")),
    ("escalar", StandardScaler())
])

# Tratamiento de variables categóricas
preprocesamiento_categorico = Pipeline([
    ("imputar", SimpleImputer(strategy="most_frequent")),
    ("codificar", OneHotEncoder(handle_unknown="ignore"))
])

# Preprocesador general
preprocesador = ColumnTransformer([
    ("numericas", preprocesamiento_numerico, columnas_numericas),
    ("categoricas", preprocesamiento_categorico, columnas_categoricas)
])

print("Variables utilizadas:", X.columns.tolist())
print("Variable objetivo: Transported")
print("Dimensiones:", X.shape)

Variables utilizadas: ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
Variable objetivo: Transported
Dimensiones: (8693, 10)


C:\Users\hp\AppData\Local\Temp\ipykernel_15524\4292392371.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_categoricas = X.select_dtypes(


**Perform Train Test Split**

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Entrenamiento: (6954, 10)
Prueba: (1739, 10)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [6]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

resultados = []

for nombre, usar_bootstrap in [
    ("Bagging", True),
    ("Pasting", False)
]:
    modelo = Pipeline([
        ("preprocesamiento", preprocesador),
        ("clasificador", BaggingClassifier(
            estimator=DecisionTreeClassifier(
                max_depth=10,
                random_state=42
            ),
            n_estimators=100,
            max_samples=0.8,
            bootstrap=usar_bootstrap,
            random_state=42,
            n_jobs=-1
        ))
    ])

    modelo.fit(X_train, y_train)
    predicciones = modelo.predict(X_test)

    accuracy = accuracy_score(y_test, predicciones)
    f1 = f1_score(y_test, predicciones)

    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy,
        "F1": f1
    })

    print(nombre)
    print("Accuracy:", round(accuracy, 4))
    print("F1:", round(f1, 4))
    print()

Bagging
Accuracy: 0.7959
F1: 0.7973

Pasting
Accuracy: 0.7895
F1: 0.7899



- Random Forests

In [7]:
from sklearn.ensemble import RandomForestClassifier

modelo_rf = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ))
])

modelo_rf.fit(X_train, y_train)

predicciones_rf = modelo_rf.predict(X_test)

accuracy_rf = accuracy_score(y_test, predicciones_rf)
f1_rf = f1_score(y_test, predicciones_rf)

resultados.append({
    "Modelo": "Random Forest",
    "Accuracy": accuracy_rf,
    "F1": f1_rf
})

print("Random Forest")
print("Accuracy:", round(accuracy_rf, 4))
print("F1:", round(f1_rf, 4))

Random Forest
Accuracy: 0.7959
F1: 0.7977


- Gradient Boosting

In [8]:
from sklearn.ensemble import GradientBoostingClassifier

modelo_gb = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

modelo_gb.fit(X_train, y_train)

predicciones_gb = modelo_gb.predict(X_test)

accuracy_gb = accuracy_score(y_test, predicciones_gb)
f1_gb = f1_score(y_test, predicciones_gb)

resultados.append({
    "Modelo": "Gradient Boosting",
    "Accuracy": accuracy_gb,
    "F1": f1_gb
})

print("Gradient Boosting")
print("Accuracy:", round(accuracy_gb, 4))
print("F1:", round(f1_gb, 4))

Gradient Boosting
Accuracy: 0.8033
F1: 0.8121


- Adaptive Boosting

In [9]:
from sklearn.ensemble import AdaBoostClassifier

modelo_ada = Pipeline([
    ("preprocesamiento", preprocesador),
    ("clasificador", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(
            max_depth=1,
            random_state=42
        ),
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ))
])

modelo_ada.fit(X_train, y_train)

predicciones_ada = modelo_ada.predict(X_test)

accuracy_ada = accuracy_score(y_test, predicciones_ada)
f1_ada = f1_score(y_test, predicciones_ada)

resultados.append({
    "Modelo": "AdaBoost",
    "Accuracy": accuracy_ada,
    "F1": f1_ada
})

print("AdaBoost")
print("Accuracy:", round(accuracy_ada, 4))
print("F1:", round(f1_ada, 4))

AdaBoost
Accuracy: 0.7458
F1: 0.7174


Which model is the best and why?

In [10]:
tabla_resultados = pd.DataFrame(resultados)

tabla_resultados = tabla_resultados.sort_values(
    by="F1",
    ascending=False
)

display(tabla_resultados.round(4))

mejor_modelo = tabla_resultados.iloc[0]

print("Mejor modelo según F1:", mejor_modelo["Modelo"])
print("F1:", round(mejor_modelo["F1"], 4))
print("Accuracy:", round(mejor_modelo["Accuracy"], 4))

,Modelo,Accuracy,F1
3,Gradient Boosting,0.8033,0.8121
2,Random Forest,0.7959,0.7977
0,Bagging,0.7959,0.7973
1,Pasting,0.7895,0.7899
4,AdaBoost,0.7458,0.7174


Mejor modelo según F1: Gradient Boosting
F1: 0.8121
Accuracy: 0.8033


### Conclusión

En este laboratorio hemos comparado cinco métodos ensemble: Bagging, Pasting, Random Forest, Gradient Boosting y AdaBoost.

El modelo que ha obtenido mejores resultados es **Gradient Boosting**, con una accuracy de **0.8033 (80,33 %)** y un F1-score de **0.8121**.

Gradient Boosting construye árboles de forma secuencial, de manera que cada nuevo árbol intenta corregir los errores de los anteriores. Esto le permite aprender patrones complejos y mejorar progresivamente sus predicciones.

Random Forest también ha obtenido buenos resultados, con un F1 de 0.7977, aunque ligeramente inferior al de Gradient Boosting.

Por tanto, seleccionamos **Gradient Boosting** como el mejor modelo de los probados, ya que presenta el F1-score más alto y también la mayor accuracy en el conjunto de prueba.